In [1]:
# imports
import pymupdf

In [37]:
# what do we need to get as input files?
INPUT_TEX_FILE = "../../files/data-science-book_book/outputs/data-science-book_pg_sep_bib.tex"
BOOK_PDF_FILE = "../../files/data-science-book_book/inputs/data-science-book.pdf"
OUTPUT_TEX_FILE = "../../files/data-science-book_book/outputs/data-science-book_pg_sep_testing_cleaned.tex"

In [9]:
# read basic tex file

tex_file = ""
with open(INPUT_TEX_FILE, "r") as file:
    tex_file = file.read()      # Read the entire file
    # print(tex_file[1000:1500])  # Print the first 500 characters to verify
    print(f"Length of tex file: {len(tex_file)} characters")

# read pdf file
pdf_document = pymupdf.open(BOOK_PDF_FILE)
print(f"Number of pages in PDF: {pdf_document.page_count}")

Length of tex file: 1103412 characters
Number of pages in PDF: 456


In [46]:
# extract toc from the pdf file
toc = pdf_document.get_toc()
print(f"Extracted TOC with {len(toc)} entries")

Extracted TOC with 302 entries


In [47]:
toc

[[1, 'Preface', 6],
 [1, 'Contents', 12],
 [1, '1\rWhat is Data Science?', 19],
 [2, '1.1 Computer Science, Data Science, and Real', 20],
 [2, 'Science', 20],
 [2, '1.2 Asking Interesting Questions from Data', 22],
 [3, '1.2.1 The Baseball Encyclopedia', 23],
 [3, '1.2.2 The Internet Movie Database (IMDb)', 25],
 [3, '1.2.3 Google Ngrams', 28],
 [3, '1.2.4 New York Taxi Records', 29],
 [2, '1.3 Properties of Data', 32],
 [3, '1.3.1 Structured vs. Unstructured Data', 32],
 [3, '1.3.2 Quantitative vs. Categorical Data', 33],
 [3, '1.3.3 Big Data vs. Little Data', 33],
 [2, '1.4 Classi\x0ccation and Regression', 34],
 [2, '1.5 Data Science Television: The Quant Shop', 35],
 [3, '1.5.1 Kaggle Challenges', 37],
 [2, '1.6 About the War Stories', 37],
 [2, '1.7 War Story: Answering the Right Question', 39],
 [2, '1.8 Chapter Notes', 40],
 [2, '1.9 Exercises', 41],
 [1, '2\rMathematical Preliminaries', 44],
 [2, '2.1 Probability', 44],
 [3, '2.1.1 Probability vs. Statistics', 46],
 [3, '2.1.2 

In [ ]:
# Cleaning things
# 1. Remove * from section titles - done
# 2. Remove starting numbers from section titles - done
# 3. Remove empty sections or chapters - done
# 4. Configure TOC -
# 5. Remove the contents section from the tex file and replace with the toc command


In [ ]:
# 1. Remove * from section titles

import re
def remove_star_from_sectioning(input_tex: str) -> str:
    """
    Removes the * from \chapter*, \section*, \subsection*, \subsubsection*
    and converts them to their non-starred versions.
    """
    pattern = r'\\(chapter|section|subsection|subsubsection)\*\{(.*?)\}'
    replacement = r'\\\1{\2}'
    return re.sub(pattern, replacement, input_tex, flags=re.DOTALL)

In [24]:
input_tex = r"""
\chapter*{Overview}
\section*{Preface}
\subsection*{Motivation}
\subsubsection*{Details}
"""

print(remove_star_from_sectioning(input_tex))


\chapter{Overview}
\section{Preface}
\subsection{Motivation}
\subsubsection{Details}



In [ ]:
# 2. Remove initial numbers from section titles

def remove_leading_numbers_from_headings(input_tex: str) -> str:
    """
    Removes leading numbers (like '7.10 ') from chapter/section/subsection/subsubsection titles.
    """
    pattern = r'(\\(?:chapter|section|subsection|subsubsection)\{)\s*\d+(?:\.\d+)*\s*(.*?)\}'
    replacement = r'\1\2}'
    return re.sub(pattern, replacement, input_tex)

In [23]:
input_tex = r"""
\chapter{1 Introduction}
\section{3.2 Background}
\subsection{7.10 Exercises}
\subsubsection{2.4.1 Detailed Analysis}
"""

print(remove_leading_numbers_from_headings(input_tex))


\chapter{Introduction}
\section{Background}
\subsection{Exercises}
\subsubsection{Detailed Analysis}



In [ ]:
# 3. Remove empty sections or chapters

def remove_empty_headings(input_tex: str) -> str:
    """
    Removes chapter/section/subsection/subsubsection headings that have no
    actual text content before the next heading or end of file.
    """
    # Split the tex into (heading, content) pairs
    pattern = r'(\\(?:chapter|section|subsection|subsubsection)\{.*?\})(.*?)(?=(\\(?:chapter|section|subsection|subsubsection)\{|$))'
    
    def replacer(match):
        heading = match.group(1)
        content = match.group(2)
        # If content is only whitespace/newlines, drop this heading block
        if content.strip() == '':
            return ''
        return heading + content

    return re.sub(pattern, replacer, input_tex, flags=re.DOTALL)

In [22]:
input_tex = r"""
\chapter{Section A}

\section{Section B}
Some text here

\subsection{Sub A}

\subsection{Sub B}
More content
"""

print(remove_empty_headings(input_tex))


\section{Section B}
Some text here

\subsection{Sub B}
More content



In [ ]:
# working with TOC step 4 onwards

In [ ]:
import fitz

# extract a good quality TOC from the PDF file

def get_cleaned_toc(pdf_path: str):
    """
    Extracts the table of contents from a PDF and removes leading numbers
    from the titles (like '1.2.3 ' or '1\\r').
    """
    doc = fitz.open(pdf_path)
    toc = doc.get_toc()
    
    cleaned_toc = []
    for level, title, page in toc:
        # Clean \r and other weird whitespace
        title = title.replace('\r', ' ').strip()
        # Remove leading numbers + dots (like "1.2.3 " or "7 ")
        title = re.sub(r'^\s*\d+(?:\.\d+)*\s*', '', title)
        cleaned_toc.append([level, title, page])
    
    return cleaned_toc

In [27]:
toc = get_cleaned_toc(BOOK_PDF_FILE)
for entry in toc:
    print(entry)

[1, 'Preface', 6]
[1, 'Contents', 12]
[1, 'What is Data Science?', 19]
[2, 'Computer Science, Data Science, and Real', 20]
[2, 'Science', 20]
[2, 'Asking Interesting Questions from Data', 22]
[3, 'The Baseball Encyclopedia', 23]
[3, 'The Internet Movie Database (IMDb)', 25]
[3, 'Google Ngrams', 28]
[3, 'New York Taxi Records', 29]
[2, 'Properties of Data', 32]
[3, 'Structured vs. Unstructured Data', 32]
[3, 'Quantitative vs. Categorical Data', 33]
[3, 'Big Data vs. Little Data', 33]
[2, 'Classi\x0ccation and Regression', 34]
[2, 'Data Science Television: The Quant Shop', 35]
[3, 'Kaggle Challenges', 37]
[2, 'About the War Stories', 37]
[2, 'War Story: Answering the Right Question', 39]
[2, 'Chapter Notes', 40]
[2, 'Exercises', 41]
[1, 'Mathematical Preliminaries', 44]
[2, 'Probability', 44]
[3, 'Probability vs. Statistics', 46]
[3, 'Compound Events and Independence', 47]
[3, 'Conditional Probability', 48]
[3, 'Probability Distributions', 49]
[2, 'Descriptive Statistics', 51]
[3, 'Centr

In [ ]:
# fix latex file with the chapters

def fix_latex_headings(input_tex: str, cleaned_toc: list) -> str:
    """
    Fix LaTeX headings based on cleaned TOC and print changes.

    Parameters:
        input_tex (str): The LaTeX document as a string.
        cleaned_toc (list): List of tuples [(level, title, page), ...] from TOC.

    Returns:
        str: Corrected LaTeX string.
    """
    # Build a mapping from TOC title -> level
    toc_map = {title: level for (level, title, _) in cleaned_toc}

    # Regex to match all LaTeX headings
    heading_pattern = re.compile(r'\\(chapter|section|subsection|subsubsection)\{(.*?)\}')

    def correct_heading(match):
        current_cmd = match.group(1)      # current LaTeX command
        title = match.group(2).strip()    # heading title

        # Check if this title exists in TOC
        if title in toc_map:
            toc_level = toc_map[title]

            # Map TOC level to LaTeX command
            level_to_cmd = {1: 'chapter', 2: 'section', 3: 'subsection'}
            correct_cmd = level_to_cmd.get(toc_level, current_cmd)

            if correct_cmd != current_cmd:
                print(f"Modified Heading :: {title} : {current_cmd} -> {correct_cmd}")
                return f'\\{correct_cmd}{{{title}}}'

        return match.group(0)  # no change

    # Replace all headings with corrected ones
    corrected_tex = heading_pattern.sub(correct_heading, input_tex)
    return corrected_tex

In [ ]:
# THIS IS WORNG
def replace_contents_section_with_toc(input_tex: str) -> str:
    """
    Replace the manually added 'Contents' section/chapter with LaTeX's \tableofcontents.
    
    Parameters:
        input_tex (str): The LaTeX document as a string.
    
    Returns:
        str: LaTeX string with 'Contents' replaced by \tableofcontents
    """
    # Pattern to match chapter/section/subsection with title "Contents"
    # It captures from the heading to just before the next heading
    pattern = re.compile(
        r'(\\(chapter|section|subsection|subsubsection)\{.*?[Cc]ontents.*?\})'  # heading
        r'(.*?)'  # content until next heading
        r'(?=\\(chapter|section|subsection|subsubsection)\{)',  # lookahead for next heading
        re.DOTALL
    )

    # Replace matched content with \tableofcontents
    def replacer(match):
        heading = match.group(1)
        print(f"\n\nRemoved '{heading}' and replaced with \\tableofcontents")
        return "\\tableofcontents\n\n"

    new_tex = pattern.sub(replacer, input_tex, count=1)  # only first occurrence
    return new_tex


def replace_first_contents_with_toc_2(input_tex: str) -> str:
    """
    Replace the first chapter/section/subsection whose title contains 'content' or 'contents'
    with \tableofcontents.
    Only removes the section starting from that heading until the next heading of the same or higher level.
    """
    # Match headings and capture level
    heading_pattern = re.compile(
        r'\\(chapter|section|subsection|subsubsection)\{.*?(content|contents).*?\}',
        re.IGNORECASE
    )

    match = heading_pattern.search(input_tex)
    if not match:
        # No "Contents"-like heading found
        return input_tex

    start_idx = match.start()
    end_idx = match.end()
    heading_level = match.group(1)  # chapter, section, etc.

    # Map LaTeX headings to numeric levels
    level_map = {'chapter': 1, 'section': 2, 'subsection': 3, 'subsubsection': 4}
    this_level = level_map.get(heading_level, 2)  # default to 2 if unknown

    # Find the next heading of same or higher level after this match
    next_heading_pattern = re.compile(
        r'\\(chapter|section|subsection|subsubsection)\{',
        re.IGNORECASE
    )

    next_matches = list(next_heading_pattern.finditer(input_tex, end_idx))
    for nm in next_matches:
        next_level = level_map.get(nm.group(1).lower(), 2)
        if next_level <= this_level:
            end_idx = nm.start()
            break
    else:
        # No next heading found; remove until the end
        end_idx = len(input_tex)

    # Replace the "Contents" section with \tableofcontents
    print(f"Removed '{input_tex[match.start():end_idx][:60]}...' and replaced with \\tableofcontents")
    new_tex = input_tex[:start_idx] + "\\tableofcontents\n\n" + input_tex[end_idx:]
    return new_tex


# THIS IS NEW AND RISKY
def replace_first_contents_with_toc(input_tex: str) -> str:
    """
    Replace the first chapter/section/subsection whose title contains 'content' or 'contents'
    with \tableofcontents.

    Parameters:
        input_tex (str): LaTeX document as a string.

    Returns:
        str: LaTeX string with the first 'Contents'-like heading replaced by \tableofcontents.
    """
    # Pattern to match headings containing 'content' or 'contents'
    pattern = re.compile(
        r'(\\(chapter|section|subsection|subsubsection)\{.*?(content|contents).*?\})'  # heading
        r'(.*?)'  # content until next heading
        r'(?=\\(chapter|section|subsection|subsubsection)\{)',  # lookahead for next heading
        re.DOTALL | re.IGNORECASE
    )

    def replacer(match):
        heading = match.group(1)
        print(f"\n\nRemoved '{heading}' and replaced with \\tableofcontents")
        return "\\tableofcontents\n\n"

    # Replace only the first occurrence
    new_tex = pattern.sub(replacer, input_tex, count=1)
    return new_tex

In [34]:
input_tex = r"""
\documentclass{book}

\begin{document}

\chapter{Contents}
This is the manually added table of contents.
It lists chapters and sections as plain text.

\chapter{Introduction}
Some introduction text.
\chapter{Background}
Some background text.
\section{Methods}
Description of methods.
\chapter{Results}
Results go here.

\end{document}
"""

cleaned_tex = replace_contents_section_with_toc(input_tex)
print(cleaned_tex)

cleaned_tex = replace_first_contents_with_toc(input_tex)
print(cleaned_tex)

Removed '\chapter{Contents}' and replaced with \tableofcontents

\documentclass{book}

\begin{document}

\tableofcontents

\chapter{Introduction}
Some introduction text.
\chapter{Background}
Some background text.
\section{Methods}
Description of methods.
\chapter{Results}
Results go here.

\end{document}

Removed '\chapter{Contents}' and replaced with \tableofcontents

\documentclass{book}

\begin{document}

\tableofcontents

\chapter{Introduction}
Some introduction text.
\chapter{Background}
Some background text.
\section{Methods}
Description of methods.
\chapter{Results}
Results go here.

\end{document}



In [36]:
input_tex = r"""
\documentclass{book}
\begin{document}

\chapter{Contents}
This is manually added table of contents.
It lists chapters and sections as plain text.

\section{Introduction}
Some introduction text.

\chapter{Background}
Some background text.

\section{Contents of Appendix}  % <-- This will NOT be removed
Some text about appendix.

\end{document}
"""

cleaned_tex = replace_contents_section_with_toc(input_tex)
print(cleaned_tex)

cleaned_tex = replace_first_contents_with_toc(input_tex)
print(cleaned_tex)

Removed '\chapter{Contents}' and replaced with \tableofcontents

\documentclass{book}
\begin{document}

\tableofcontents

\section{Introduction}
Some introduction text.

\chapter{Background}
Some background text.

\section{Contents of Appendix}  % <-- This will NOT be removed
Some text about appendix.

\end{document}

Removed '\chapter{Contents}' and replaced with \tableofcontents

\documentclass{book}
\begin{document}

\tableofcontents

\section{Introduction}
Some introduction text.

\chapter{Background}
Some background text.

\section{Contents of Appendix}  % <-- This will NOT be removed
Some text about appendix.

\end{document}



In [ ]:
def driver():
    INPUT_TEX_FILE = "../../files/data-science-book_book/outputs/data-science-book_pg_sep_bib.tex"
    BOOK_PDF_FILE = "../../files/data-science-book_book/inputs/data-science-book.pdf"
    OUTPUT_TEX_FILE = "../../files/data-science-book_book/outputs/data-science-book_pg_sep_testing_cleaned.tex"

    tex_file = ""
    with open(INPUT_TEX_FILE, "r") as file:
        tex_file = file.read()      # Read the entire file
        # print(tex_file[1000:1500])  # Print the first 500 characters to verify
        print(f"Length of tex file: {len(tex_file)} characters")

    unstared_tex = remove_star_from_sectioning(tex_file)
    print("Removed * from section titles")
    no_numbers_tex = remove_leading_numbers_from_headings(unstared_tex)
    print("Removed leading numbers from section titles")
    cleaned_tex = remove_empty_headings(no_numbers_tex)
    print("Removed empty sections/chapters from the tex file")

    # get cleaned toc
    toc = get_cleaned_toc(BOOK_PDF_FILE)
    print(f"Extracted TOC with {len(toc)} entries")

    fixed_headings_tex = fix_latex_headings(cleaned_tex, toc)
    print("Fixed LaTeX headings based on cleaned TOC")
    final_tex = replace_first_contents_with_toc_2(fixed_headings_tex)
    print("Replaced 'Contents' section/chapter with \\tableofcontents")

    # Write the final cleaned LaTeX to the output file
    with open(OUTPUT_TEX_FILE, "w") as out_file:
        out_file.write(final_tex)
    print(f"Final cleaned LaTeX written to: {OUTPUT_TEX_FILE}")





In [45]:
driver()

Length of tex file: 1103412 characters
Removed * from section titles
Removed leading numbers from section titles
Removed empty sections/chapters from the tex file
Extracted TOC with 302 entries
Preface : section -> chapter
Contents : section -> chapter
What is Data Science? : section -> chapter
Asking Interesting Questions from Data : subsection -> section
Properties of Data : subsection -> section
Data Science Television: The Quant Shop : subsection -> section
About the War Stories : subsection -> section
War Story: Answering the Right Question : subsection -> section
Chapter Notes : subsection -> section
Kaggle Challenges : section -> subsection
Mathematical Preliminaries : section -> chapter
Probability : subsection -> section
Descriptive Statistics : subsection -> section
Correlation Analysis : subsection -> section
Logarithms : subsection -> section
War Story: Fitting Designer Genes : subsection -> section
Chapter Notes : subsection -> section
Kaggle Challenges : section -> subsec